In [0]:
%sql
-- Gold Table 1: Content by Release Year
-- Analyzes yearly content trends with Movie vs TV Show breakdown

CREATE OR REPLACE TABLE data_engineering_projects.netflix_data.netflix_content_by_year_gold AS
SELECT 
  release_year,
  COUNT(*) as total_content,
  SUM(CASE WHEN type = 'Movie' THEN 1 ELSE 0 END) as movie_count,
  SUM(CASE WHEN type = 'TV Show' THEN 1 ELSE 0 END) as tv_show_count,
  ROUND(SUM(CASE WHEN type = 'Movie' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as movie_percentage,
  ROUND(SUM(CASE WHEN type = 'TV Show' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as tv_show_percentage,
  ROUND(AVG(data_quality_score), 2) as avg_quality_score,
  MIN(date_added) as earliest_added_date,
  MAX(date_added) as latest_added_date,
  current_timestamp() as created_at
FROM data_engineering_projects.netflix_data.netflix_data_silver
WHERE release_year IS NOT NULL
GROUP BY release_year
ORDER BY release_year DESC;

SELECT * FROM data_engineering_projects.netflix_data.netflix_content_by_year_gold LIMIT 10;

release_year,total_content,movie_count,tv_show_count,movie_percentage,tv_show_percentage,avg_quality_score,earliest_added_date,latest_added_date,created_at
2021,589,275,314,46.69,53.31,91.58,2020-02-13,2021-09-24,2026-08-21T15:50:32.066Z
2020,952,516,436,54.20,45.80,93.89,2019-10-11,2021-09-25,2026-08-21T15:50:32.066Z
2019,1026,630,396,61.40,38.60,94.43,2018-05-29,2021-09-10,2026-08-21T15:50:32.066Z
2018,1145,765,380,66.81,33.19,95.07,2016-12-23,2021-09-22,2026-08-21T15:50:32.066Z
2017,1029,764,265,74.25,25.75,96.0,2016-12-13,2021-09-16,2026-08-21T15:50:32.066Z
2016,901,657,244,72.92,27.08,95.78,2013-03-31,2021-09-06,2026-08-21T15:50:32.066Z
2015,558,396,162,70.97,29.03,95.86,2015-01-23,2021-09-15,2026-08-21T15:50:32.066Z
2014,352,264,88,75.00,25.00,96.63,2014-01-24,2021-09-15,2026-08-21T15:50:32.066Z
2013,288,225,63,78.13,21.88,96.75,2013-08-02,2021-09-19,2026-08-21T15:50:32.066Z
2012,237,173,64,73.00,27.00,96.5,2012-11-14,2021-09-16,2026-08-21T15:50:32.066Z


In [0]:
%sql
-- Gold Table 2: Genre Analysis
-- Explodes and analyzes genre distribution across content

CREATE OR REPLACE TABLE data_engineering_projects.netflix_data.netflix_genre_analysis_gold AS
WITH exploded_genres AS (
  SELECT 
    show_id,
    type,
    release_year,
    rating,
    TRIM(genre_item.col) as genre,
    data_quality_score
  FROM data_engineering_projects.netflix_data.netflix_data_silver
  LATERAL VIEW explode(split(genres, ',')) genre_item
  WHERE genres IS NOT NULL
)
SELECT 
  genre,
  COUNT(*) as total_content,
  SUM(CASE WHEN type = 'Movie' THEN 1 ELSE 0 END) as movie_count,
  SUM(CASE WHEN type = 'TV Show' THEN 1 ELSE 0 END) as tv_show_count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as percentage_of_total,
  ROUND(AVG(data_quality_score), 2) as avg_quality_score,
  MIN(release_year) as earliest_release_year,
  MAX(release_year) as latest_release_year,
  ROUND(AVG(release_year), 0) as avg_release_year,
  current_timestamp() as created_at
FROM exploded_genres
GROUP BY genre
ORDER BY total_content DESC;

SELECT * FROM data_engineering_projects.netflix_data.netflix_genre_analysis_gold LIMIT 10;

genre,total_content,movie_count,tv_show_count,percentage_of_total,avg_quality_score,earliest_release_year,latest_release_year,avg_release_year,created_at
International Movies,2748,2748,0,14.26,98.8,1954,2021,2014.0,2026-08-21T15:51:33.870Z
Dramas,2419,2419,0,12.55,99.62,1954,2021,2013.0,2026-08-21T15:51:33.870Z
Comedies,1670,1670,0,8.66,99.46,1954,2021,2012.0,2026-08-21T15:51:33.870Z
International TV Shows,1350,0,1350,7.00,89.3,1972,2021,2017.0,2026-08-21T15:51:33.870Z
Documentaries,866,866,0,4.49,94.02,1942,2021,2015.0,2026-08-21T15:51:33.870Z
Action & Adventure,857,857,0,4.45,99.59,1956,2021,2010.0,2026-08-21T15:51:33.870Z
TV Dramas,762,0,762,3.95,90.03,1986,2021,2017.0,2026-08-21T15:51:33.870Z
Independent Movies,751,751,0,3.90,99.82,1955,2021,2014.0,2026-08-21T15:51:33.870Z
Children & Family Movies,641,641,0,3.33,98.1,1954,2021,2014.0,2026-08-21T15:51:33.870Z
Romantic Movies,616,616,0,3.20,99.48,1960,2021,2013.0,2026-08-21T15:51:33.870Z


In [0]:
%sql
-- Gold Table 3: Country Production Analysis
-- Explodes and analyzes which countries produce most content

CREATE OR REPLACE TABLE data_engineering_projects.netflix_data.netflix_country_production_gold AS
WITH exploded_countries AS (
  SELECT 
    show_id,
    type,
    release_year,
    rating,
    TRIM(country_item.col) as country,
    data_quality_score
  FROM data_engineering_projects.netflix_data.netflix_data_silver
  LATERAL VIEW explode(split(country, ',')) country_item
  WHERE country IS NOT NULL
)
SELECT 
  country,
  COUNT(*) as total_content,
  SUM(CASE WHEN type = 'Movie' THEN 1 ELSE 0 END) as movie_count,
  SUM(CASE WHEN type = 'TV Show' THEN 1 ELSE 0 END) as tv_show_count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as percentage_of_total,
  ROUND(AVG(data_quality_score), 2) as avg_quality_score,
  MIN(release_year) as earliest_release_year,
  MAX(release_year) as latest_release_year,
  ROUND(AVG(release_year), 0) as avg_release_year,
  COUNT(DISTINCT release_year) as years_of_production,
  current_timestamp() as created_at
FROM exploded_countries
GROUP BY country
ORDER BY total_content DESC;

SELECT * FROM data_engineering_projects.netflix_data.netflix_country_production_gold LIMIT 10;

country,total_content,movie_count,tv_show_count,percentage_of_total,avg_quality_score,earliest_release_year,latest_release_year,avg_release_year,years_of_production,created_at
United States,3673,2737,936,36.75,96.32,1942,2021,2013.0,70,2026-08-21T15:51:47.540Z
India,1046,962,84,10.47,98.81,1959,2021,2012.0,57,2026-08-21T15:51:47.540Z
United Kingdom,805,533,272,8.05,95.46,1944,2021,2014.0,45,2026-08-21T15:51:47.540Z
Canada,445,319,126,4.45,96.43,1979,2021,2015.0,29,2026-08-21T15:51:47.540Z
France,392,302,90,3.92,97.09,1955,2021,2014.0,35,2026-08-21T15:51:47.540Z
Japan,318,119,199,3.18,93.76,1979,2021,2014.0,33,2026-08-21T15:51:47.540Z
South Korea,231,61,170,2.31,92.86,2004,2021,2017.0,15,2026-08-21T15:51:47.540Z
Spain,230,169,61,2.30,96.78,1993,2021,2017.0,19,2026-08-21T15:51:47.540Z
Germany,224,180,44,2.24,97.1,1993,2021,2014.0,26,2026-08-21T15:51:47.540Z
Mexico,169,111,58,1.69,95.86,1979,2021,2016.0,24,2026-08-21T15:51:47.540Z


In [0]:
%sql
-- Gold Table 4: Rating Analysis
-- Analyzes content maturity rating patterns

CREATE OR REPLACE TABLE data_engineering_projects.netflix_data.netflix_rating_analysis_gold AS
SELECT 
  rating,
  COUNT(*) as total_content,
  SUM(CASE WHEN type = 'Movie' THEN 1 ELSE 0 END) as movie_count,
  SUM(CASE WHEN type = 'TV Show' THEN 1 ELSE 0 END) as tv_show_count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as percentage_of_total,
  ROUND(AVG(data_quality_score), 2) as avg_quality_score,
  MIN(release_year) as earliest_release_year,
  MAX(release_year) as latest_release_year,
  ROUND(AVG(release_year), 0) as avg_release_year,
  -- Categorize ratings into maturity levels
  CASE 
    WHEN rating IN ('G', 'TV-Y', 'TV-G') THEN 'All Ages'
    WHEN rating IN ('PG', 'TV-Y7', 'TV-PG') THEN 'Parental Guidance'
    WHEN rating IN ('PG-13', 'TV-14') THEN 'Teens 13+'
    WHEN rating IN ('R', 'TV-MA', 'NC-17') THEN 'Mature 17+'
    WHEN rating IN ('NR', 'UR') THEN 'Not Rated'
    ELSE 'Other'
  END as maturity_category,
  current_timestamp() as created_at
FROM data_engineering_projects.netflix_data.netflix_data_silver
WHERE rating IS NOT NULL
GROUP BY rating
ORDER BY total_content DESC;

SELECT * FROM data_engineering_projects.netflix_data.netflix_rating_analysis_gold LIMIT 10;

rating,total_content,movie_count,tv_show_count,percentage_of_total,avg_quality_score,earliest_release_year,latest_release_year,avg_release_year,maturity_category,created_at
TV-MA,3195,2052,1143,36.39,95.13,1945,2021,2017.0,Mature 17+,2026-08-21T15:52:00.027Z
TV-14,2158,1426,732,24.58,95.24,1925,2021,2014.0,Teens 13+,2026-08-21T15:52:00.027Z
TV-PG,862,539,323,9.82,94.01,1943,2021,2014.0,Parental Guidance,2026-08-21T15:52:00.027Z
R,796,794,2,9.07,99.77,1962,2021,2010.0,Mature 17+,2026-08-21T15:52:00.027Z
PG-13,489,489,0,5.57,99.63,1955,2021,2009.0,Teens 13+,2026-08-21T15:52:00.027Z
TV-Y7,334,139,195,3.80,91.72,1981,2021,2016.0,Parental Guidance,2026-08-21T15:52:00.027Z
TV-Y,307,131,176,3.50,91.03,1992,2021,2017.0,All Ages,2026-08-21T15:52:00.027Z
PG,286,286,0,3.26,99.58,1973,2021,2008.0,Parental Guidance,2026-08-21T15:52:00.027Z
TV-G,220,126,94,2.51,93.0,1954,2021,2016.0,All Ages,2026-08-21T15:52:00.027Z
NR,80,75,5,0.91,97.06,1958,2018,2011.0,Not Rated,2026-08-21T15:52:00.027Z


In [0]:
%sql
-- Gold Table 5: Monthly Trends
-- Analyzes content addition patterns over time by month

CREATE OR REPLACE TABLE data_engineering_projects.netflix_data.netflix_monthly_trends_gold AS
SELECT 
  YEAR(date_added) as add_year,
  MONTH(date_added) as add_month,
  DATE_TRUNC('MONTH', date_added) as month_date,
  COUNT(*) as total_content_added,
  SUM(CASE WHEN type = 'Movie' THEN 1 ELSE 0 END) as movies_added,
  SUM(CASE WHEN type = 'TV Show' THEN 1 ELSE 0 END) as tv_shows_added,
  ROUND(AVG(data_quality_score), 2) as avg_quality_score,
  -- Rolling 3-month average
  ROUND(AVG(COUNT(*)) OVER (
    ORDER BY DATE_TRUNC('MONTH', date_added)
    ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
  ), 2) as rolling_3month_avg,
  -- Year-over-year comparison
  LAG(COUNT(*), 12) OVER (ORDER BY DATE_TRUNC('MONTH', date_added)) as same_month_prior_year,
  COUNT(DISTINCT rating) as distinct_ratings_added,
  current_timestamp() as created_at
FROM data_engineering_projects.netflix_data.netflix_data_silver
WHERE date_added IS NOT NULL
GROUP BY YEAR(date_added), MONTH(date_added), DATE_TRUNC('MONTH', date_added)
ORDER BY add_year DESC, add_month DESC;

SELECT * FROM data_engineering_projects.netflix_data.netflix_monthly_trends_gold LIMIT 10;

add_year,add_month,month_date,total_content_added,movies_added,tv_shows_added,avg_quality_score,rolling_3month_avg,same_month_prior_year,distinct_ratings_added,created_at
2021,9,2021-09-01T00:00:00.000Z,181,116,65,94.28,205.0,168,10,2026-08-21T15:52:13.016Z
2021,8,2021-08-01T00:00:00.000Z,177,117,60,94.55,213.33,128,9,2026-08-21T15:52:13.016Z
2021,7,2021-07-01T00:00:00.000Z,257,169,88,93.91,198.33,145,9,2026-08-21T15:52:13.016Z
2021,6,2021-06-01T00:00:00.000Z,206,123,83,94.15,175.33,156,9,2026-08-21T15:52:13.016Z
2021,5,2021-05-01T00:00:00.000Z,132,94,38,94.66,143.67,157,10,2026-08-21T15:52:13.016Z
2021,4,2021-04-01T00:00:00.000Z,188,135,53,94.97,136.0,176,10,2026-08-21T15:52:13.016Z
2021,3,2021-03-01T00:00:00.000Z,111,74,37,95.41,116.67,137,9,2026-08-21T15:52:13.016Z
2021,2,2021-02-01T00:00:00.000Z,109,65,44,94.91,135.67,114,8,2026-08-21T15:52:13.016Z
2021,1,2021-01-01T00:00:00.000Z,130,94,36,96.23,150.33,204,10,2026-08-21T15:52:13.016Z
2020,12,2020-12-01T00:00:00.000Z,168,101,67,95.09,162.67,215,10,2026-08-21T15:52:13.016Z


In [0]:
%sql
-- Gold Table 6: Content Quality Metrics
-- Overall data completeness and quality summary

CREATE OR REPLACE TABLE data_engineering_projects.netflix_data.netflix_content_quality_metrics_gold AS
SELECT 
  -- Overall metrics
  COUNT(*) as total_records,
  COUNT(DISTINCT show_id) as unique_shows,
  
  -- Type distribution
  SUM(CASE WHEN type = 'Movie' THEN 1 ELSE 0 END) as total_movies,
  SUM(CASE WHEN type = 'TV Show' THEN 1 ELSE 0 END) as total_tv_shows,
  
  -- Completeness metrics (percentage of non-missing values)
  ROUND((COUNT(*) - SUM(CASE WHEN is_director_missing THEN 1 ELSE 0 END)) * 100.0 / COUNT(*), 2) as director_completeness_pct,
  ROUND((COUNT(*) - SUM(CASE WHEN is_cast_missing THEN 1 ELSE 0 END)) * 100.0 / COUNT(*), 2) as cast_completeness_pct,
  ROUND((COUNT(*) - SUM(CASE WHEN is_country_missing THEN 1 ELSE 0 END)) * 100.0 / COUNT(*), 2) as country_completeness_pct,
  ROUND((COUNT(*) - SUM(CASE WHEN is_date_added_missing THEN 1 ELSE 0 END)) * 100.0 / COUNT(*), 2) as date_added_completeness_pct,
  
  -- Missing counts
  SUM(CASE WHEN is_director_missing THEN 1 ELSE 0 END) as missing_director_count,
  SUM(CASE WHEN is_cast_missing THEN 1 ELSE 0 END) as missing_cast_count,
  SUM(CASE WHEN is_country_missing THEN 1 ELSE 0 END) as missing_country_count,
  SUM(CASE WHEN is_date_added_missing THEN 1 ELSE 0 END) as missing_date_added_count,
  
  -- Quality score statistics
  ROUND(AVG(data_quality_score), 2) as avg_quality_score,
  MIN(data_quality_score) as min_quality_score,
  MAX(data_quality_score) as max_quality_score,
  ROUND(STDDEV(data_quality_score), 2) as quality_score_stddev,
  
  -- Records by quality tier
  SUM(CASE WHEN data_quality_score = 100 THEN 1 ELSE 0 END) as perfect_quality_count,
  SUM(CASE WHEN data_quality_score >= 90 AND data_quality_score < 100 THEN 1 ELSE 0 END) as high_quality_count,
  SUM(CASE WHEN data_quality_score >= 75 AND data_quality_score < 90 THEN 1 ELSE 0 END) as medium_quality_count,
  SUM(CASE WHEN data_quality_score < 75 THEN 1 ELSE 0 END) as low_quality_count,
  
  -- Data coverage period
  MIN(release_year) as earliest_release_year,
  MAX(release_year) as latest_release_year,
  MIN(date_added) as earliest_date_added,
  MAX(date_added) as latest_date_added,
  
  -- Distinct counts
  COUNT(DISTINCT release_year) as distinct_release_years,
  COUNT(DISTINCT rating) as distinct_ratings,
  COUNT(DISTINCT country) as distinct_countries_raw,
  
  current_timestamp() as created_at
FROM data_engineering_projects.netflix_data.netflix_data_silver;

SELECT * FROM data_engineering_projects.netflix_data.netflix_content_quality_metrics_gold;

total_records,unique_shows,total_movies,total_tv_shows,director_completeness_pct,cast_completeness_pct,country_completeness_pct,date_added_completeness_pct,missing_director_count,missing_cast_count,missing_country_count,missing_date_added_count,avg_quality_score,min_quality_score,max_quality_score,quality_score_stddev,perfect_quality_count,high_quality_count,medium_quality_count,low_quality_count,earliest_release_year,latest_release_year,earliest_date_added,latest_date_added,distinct_release_years,distinct_ratings,distinct_countries_raw,created_at
8786,8786,6112,2673,70.03,90.60,90.53,99.86,2633,826,832,12,95.57,60,100,6.12,5317,2736,730,3,1925,2021,2008-01-01,2021-09-25,74,14,748,2026-08-21T15:52:26.732Z
